# Lesson 02 — Gamma Correction

## Why
Gamma corrects brightness along a curve, not a line. A camera underexposing by 1 stop doesn't just need +50 brightness — it needs a non-linear lift that recovers shadows without blowing highlights.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread('sample.jpg')

def gamma_correct(img, gamma):
    # Build LUT once, apply instantly
    lut = np.array([min(255, int((i/255.0)**(1.0/gamma) * 255))
                    for i in range(256)], dtype=np.uint8)
    return cv2.LUT(img, lut)

# Show the gamma curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(256)
gammas = [0.3, 0.6, 1.0, 1.5, 2.5]
colors = ['purple','blue','black','orange','red']
for g, c in zip(gammas, colors):
    y = np.array([min(255, int((i/255.0)**(1.0/g)*255)) for i in x])
    ax1.plot(x, y, color=c, label=f'γ={g}', linewidth=2)
ax1.set_title('Gamma curves (input → output mapping)')
ax1.legend(); ax1.set_xlabel('Input'); ax1.set_ylabel('Output')
ax1.plot([0,255],[0,255],'k--',alpha=0.3)

# Show result on image
results = [gamma_correct(img, g) for g in gammas]
ax2.axis('off')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for ax, g, im in zip(axes, gammas, results):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(f'γ={g}  {"(brighter)" if g<1 else "(darker)" if g>1 else "(neutral)"}')
    ax.axis('off')
plt.show()

## Key Takeaway
gamma < 1 = brightens (lifts shadows). gamma > 1 = darkens. LUT (Look-Up Table) makes this O(1) per pixel — precompute the mapping for all 256 values once.